# 1\.1\.2 Load Taxi and FVH

This notebook loads and validates the four NYC TLC trip\-record datasets that will serve as the project's primary source of taxi and rideshare mobility activity\. Because these files contain nearly one billion trips across Yellow Taxi, Green Taxi, FHV, and FHVHV, the goal is not simply to download the data, but to establish confidence that it is complete, readable, structurally consistent, and suitable for downstream Taxi Zone\-level analysis\. Along the way, we inspect schemas, identify the key temporal and spatial fields, evaluate missingness patterns, validate geographic coverage, and make decisions about which datasets should ultimately be carried forward into the project's mobility\-state pipeline\.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import requests
import time
from urllib.error import HTTPError, URLError

import pyarrow.parquet as pq
import pandas as pd

import duckdb  # For those huge parquet files!

In [2]:
SOURCE_DATA_DIR = Path("source_data")
TLC_DATA_DIR = SOURCE_DATA_DIR / "tlc"

PIPELINE_DATA_DIR = Path("pipeline_data")

## 1\.1\.2\.1 Load the four datasets programmatically

The TLC trip record datasets form the largest mobility component of the project and provide detailed trip\-level records across multiple transportation modes\. For this analysis, we collect four datasets:

\- Yellow Taxi
\- Green Taxi
\- FHV \(traditional for\-hire vehicles such as black cars and livery services\)
\- FHVHV \(high\-volume for\-hire vehicles such as Uber and Lyft\)

To support both pre\- and post\-congestion pricing analysis, we download monthly parquet files spanning January 2023 through March 2026\. Because these datasets contain hundreds of millions of trip records and many large parquet files, we automate the download process and perform validation checks to ensure all expected files are available before beginning downstream QA and standardization work\.

In [3]:
# ---------------------------------------------------------------------
# Configure TLC trip record downloads
# ---------------------------------------------------------------------

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

TLC_DATA_DIR.mkdir(parents=True, exist_ok=True)

download_plan = {
    2023: range(1, 13),
    2024: range(1, 13),
    2025: range(1, 13),
    2026: range(1, 4),  # Jan-March only for now
}

datasets = [
    "yellow",
    "green",
    "fhv",
    "fhvhv",
]

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page",
}

First, we define the download plan and dataset configuration\. This allows us to programmatically generate every expected monthly file name and URL rather than manually downloading files one at a time\. The download plan also serves as a single source of truth for the temporal coverage included in the project\. Note that CloudFront rejects too many requests, so we had to run the cell below multiple times \(5\-10 minutes after failure\) to get all the files downloaded\.

In [4]:
# ---------------------------------------------------------------------
# Download TLC monthly parquet files
# ---------------------------------------------------------------------
# Example URLs:
# Yellow: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
# Green: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet
# FHV (black cars / limos): https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2025-03.parquet
# FHVHV (Uber/Lyft): https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-03.parquet  

for dataset in datasets:
    for year, months in download_plan.items():
        for month in months:
            month_str = f"{month:02d}"
            filename = f"{dataset}_tripdata_{year}-{month_str}.parquet"
            url = f"{BASE_URL}/{filename}"
            output_path = TLC_DATA_DIR / filename

            if output_path.exists() and output_path.stat().st_size > 0:
                print(f"Already downloaded, skipping: {filename}")
                continue

            try:
                print(f"Downloading {filename}...")

                response = requests.get(
                    url,
                    headers=headers,
                    stream=True,
                    timeout=120,
                )

                response.raise_for_status()

                with open(output_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            except requests.exceptions.HTTPError as e:
                print(f"HTTP error: {filename} -> {e}")

                if output_path.exists() and output_path.stat().st_size == 0:
                    output_path.unlink()

            except Exception as e:
                print(f"Failed: {filename} -> {e}")

Already downloaded, skipping: yellow_tripdata_2023-01.parquet
Already downloaded, skipping: yellow_tripdata_2023-02.parquet
Already downloaded, skipping: yellow_tripdata_2023-03.parquet
Already downloaded, skipping: yellow_tripdata_2023-04.parquet
Already downloaded, skipping: yellow_tripdata_2023-05.parquet
Already downloaded, skipping: yellow_tripdata_2023-06.parquet
Already downloaded, skipping: yellow_tripdata_2023-07.parquet
Already downloaded, skipping: yellow_tripdata_2023-08.parquet
Already downloaded, skipping: yellow_tripdata_2023-09.parquet
Already downloaded, skipping: yellow_tripdata_2023-10.parquet
Already downloaded, skipping: yellow_tripdata_2023-11.parquet
Already downloaded, skipping: yellow_tripdata_2023-12.parquet
Already downloaded, skipping: yellow_tripdata_2024-01.parquet
Already downloaded, skipping: yellow_tripdata_2024-02.parquet
Already downloaded, skipping: yellow_tripdata_2024-03.parquet
Already downloaded, skipping: yellow_tripdata_2024-04.parquet
Already 

The download routine skips files that already exist locally, making the notebook safe to rerun without repeatedly downloading large datasets\. Once the download process completes, we verify that all expected monthly parquet files are present and identify any missing files that may require investigation or re\-download\.

In [5]:
expected_files = []

for dataset in datasets:
    for year, months in download_plan.items():
        for month in months:
            expected_files.append(
                f"{dataset}_tripdata_{year}-{month:02d}.parquet"
            )

actual_files = {
    path.name: path
    for path in TLC_DATA_DIR.glob("*.parquet")
}

missing_files = [
    filename
    for filename in expected_files
    if filename not in actual_files
]

print("Expected files:", len(expected_files))
print("Actual parquet files:", len(actual_files))
print("Missing files:", len(missing_files))

missing_files[:50]

Expected files: 156
Actual parquet files: 157
Missing files: 0


[]

In addition to confirming file presence, it is useful to inspect file sizes across the collection\. Large deviations in file size can sometimes indicate incomplete downloads, corrupted files, schema changes, or unusual shifts in trip volume\. This quick review provides an initial sanity check before deeper parquet\-level QA\.

In [6]:
for filename in expected_files:
    path = TLC_DATA_DIR / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{filename}: {size_mb:,.1f} MB")
    else:
        print(f"MISSING: {filename}")

yellow_tripdata_2023-01.parquet: 45.5 MB
yellow_tripdata_2023-02.parquet: 45.5 MB
yellow_tripdata_2023-03.parquet: 53.5 MB
yellow_tripdata_2023-04.parquet: 51.7 MB
yellow_tripdata_2023-05.parquet: 55.9 MB
yellow_tripdata_2023-06.parquet: 52.5 MB
yellow_tripdata_2023-07.parquet: 46.1 MB
yellow_tripdata_2023-08.parquet: 45.9 MB
yellow_tripdata_2023-09.parquet: 45.7 MB
yellow_tripdata_2023-10.parquet: 56.3 MB
yellow_tripdata_2023-11.parquet: 53.5 MB
yellow_tripdata_2023-12.parquet: 54.2 MB
yellow_tripdata_2024-01.parquet: 47.6 MB
yellow_tripdata_2024-02.parquet: 48.0 MB
yellow_tripdata_2024-03.parquet: 57.3 MB
yellow_tripdata_2024-04.parquet: 56.4 MB
yellow_tripdata_2024-05.parquet: 59.7 MB
yellow_tripdata_2024-06.parquet: 57.1 MB
yellow_tripdata_2024-07.parquet: 49.9 MB
yellow_tripdata_2024-08.parquet: 48.7 MB
yellow_tripdata_2024-09.parquet: 58.3 MB
yellow_tripdata_2024-10.parquet: 61.4 MB
yellow_tripdata_2024-11.parquet: 57.8 MB
yellow_tripdata_2024-12.parquet: 58.7 MB
yellow_tripdata_

### Summary

At this point, all TLC trip record files have been downloaded and inventoried\. We have verified dataset coverage across the four transportation modes and established the expected set of monthly parquet files that will be used throughout the remainder of the pipeline\. The next step is to validate parquet health and inspect the underlying schemas before performing more detailed quality assurance checks\.

## 1\.1\.2\.2 Parquet readability QA

Before we inspect schemas or validate record\-level fields, let's confirm that every downloaded TLC parquet file can be opened successfully\.\. This serves as our first dataset health check\. We verify that each parquet file is readable, capture basic metadata such as row counts and column counts, and record any file\-level errors that might indicate incomplete downloads or corrupted files\. Because several FHVHV files are hundreds of megabytes in size, we store the results in a QA manifest\. Once the manifest has been generated, subsequent notebook runs can reload the saved results rather than rescanning every parquet file\.

In [7]:
# ---------------------------------------------------------------------
# 1.1.2.2 Parquet Health QA
# ---------------------------------------------------------------------

RUN_PARQUET_HEALTH_QA = False

QA_DIR = Path("pipeline_data") / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_HEALTH_MANIFEST_PATH = QA_DIR / "1.1.2_taxi_fhv_parquet_health_manifest.csv"

First, we configure the QA manifest location and define a rerun toggle\. This allows us to perform the parquet scan once, save the results, and avoid repeatedly opening more than 150 large parquet files during routine notebook execution\.

In [8]:
# ---------------------------------------------------------------------
# Run or load parquet health QA
# ---------------------------------------------------------------------
# This scan opens every TLC parquet footer, which is expensive across 157
# monthly files. The manifest cache lets normal reruns load prior QA results;
# set RUN_PARQUET_HEALTH_QA = True only after downloads change or if a file
# health issue needs to be rechecked from source.

if RUN_PARQUET_HEALTH_QA or not PARQUET_HEALTH_MANIFEST_PATH.exists():

    parquet_health = []

    for path in sorted(TLC_DATA_DIR.glob("*.parquet")):
        record = {
            "file": path.name,
            "size_mb": path.stat().st_size / (1024 * 1024),
            "num_rows": None,
            "num_columns": None,
            "error": None,
        }

        try:
            pf = pq.ParquetFile(path)
            record["num_rows"] = pf.metadata.num_rows
            record["num_columns"] = pf.metadata.num_columns

        except Exception as e:
            record["error"] = str(e)

        parquet_health.append(record)

    parquet_health_df = pd.DataFrame(parquet_health)

    parquet_health_df.to_csv(
        PARQUET_HEALTH_MANIFEST_PATH,
        index=False
    )

else:
    parquet_health_df = pd.read_csv(PARQUET_HEALTH_MANIFEST_PATH)

Next, we scan each parquet file and attempt to read its metadata\. For every file, we capture file size, row count, and column count while recording any exceptions encountered during the read process\. Files that fail this step will be flagged for further investigation before proceeding with downstream QA\.

In [9]:
parquet_health_df = parquet_health_df.copy()

parquet_health_df["dataset_type"] = parquet_health_df["file"].str.extract(
    r"^(.*?)_tripdata_"
)

parquet_health_df["service_month"] = parquet_health_df["file"].str.extract(
    r"_tripdata_(\d{4}-\d{2})"
)

parquet_health_df["readable"] = parquet_health_df["error"].isna()

After loading the manifest, we enrich the results with dataset\-level metadata derived from the file names\. This allows us to summarize parquet health by transportation mode and service month rather than reviewing individual files one at a time\.

In [10]:
parquet_health_summary = (
    parquet_health_df
    .groupby("dataset_type", dropna=False)
    .agg(
        num_files=("file", "count"),
        readable_files=("readable", "sum"),
        unreadable_files=("readable", lambda x: (~x).sum()),
        total_rows=("num_rows", "sum"),
        total_size_mb=("size_mb", "sum"),
        min_size_mb=("size_mb", "min"),
        max_size_mb=("size_mb", "max"),
    )
    .reset_index()
)

parquet_health_summary

,dataset_type,num_files,readable_files,unreadable_files,total_rows,total_size_mb,min_size_mb,max_size_mb
0,fhv,39,39,0,64787450,689.309167,10.785082,25.089163
1,fhvhv,39,39,0,778424569,18187.889832,429.404938,510.935586
2,green,39,39,0,2160506,49.605245,0.878098,1.650809
3,yellow,40,40,0,143080418,2302.932751,45.464869,74.231973


Findings: All 157 TLC parquet files were successfully opened and inspected, resulting in a 100% readability rate across all four datasets\. No corrupted, unreadable, or partially downloaded files were detected\. The scan also highlights the substantial scale differences across transportation modes: FHVHV dominates the collection with approximately 778 million records and 18\.2 GB of storage, followed by Yellow Taxi with approximately 143 million records and 2\.3 GB\. In comparison, FHV contains approximately 64\.8 million records and Green Taxi approximately 2\.2 million records\. These results confirm that the datasets are ready for downstream QA while also identifying FHVHV as the primary storage and memory constraint for the remainder of the pipeline\.

### Summary

The parquet readability scan confirmed that all downloaded TLC datasets could be successfully opened and inspected\. Basic file metadata, including row counts and column counts, were captured and persisted to a QA manifest for future notebook runs\. No file\-level issues requiring remediation were identified during this stage\. With parquet health validated, we can proceed to schema inspection and deeper record\-level quality assurance\.

## 1\.1\.2\.3 Schema analysis

Now that we’ve confirmed the parquet files are readable, let’s inspect the schemas across the four TLC trip record datasets\. The goal here is to understand which columns exist, how stable each dataset’s schema is across months, and which fields are likely to matter later for temporal and spatial standardization\. We are not validating record\-level quality yet; this step is about mapping the structure of the data before we start scanning values\.

In [11]:
# ---------------------------------------------------------------------
# 1.1.2.3 Schema Analysis
# ---------------------------------------------------------------------

RUN_SCHEMA_QA = False

SCHEMA_MANIFEST_PATH = QA_DIR / "1.1.2_taxi_fhv_schema_manifest.csv"

First, we create a schema manifest\. For each parquet file, we capture every column name and its data type\. Saving this as a manifest lets us compare schemas without repeatedly reopening every source file\.

In [12]:
if RUN_SCHEMA_QA or not SCHEMA_MANIFEST_PATH.exists():

    schema_records = []

    for path in sorted(TLC_DATA_DIR.glob("*.parquet")):
        dataset_type = path.name.split("_tripdata_")[0]
        service_month = path.name.split("_tripdata_")[1].replace(".parquet", "")

        try:
            pf = pq.ParquetFile(path)
            schema = pf.schema_arrow

            for field in schema:
                schema_records.append({
                    "file": path.name,
                    "dataset_type": dataset_type,
                    "service_month": service_month,
                    "column_name": field.name,
                    "data_type": str(field.type),
                })

        except Exception as e:
            schema_records.append({
                "file": path.name,
                "dataset_type": dataset_type,
                "service_month": service_month,
                "column_name": None,
                "data_type": None,
                "error": str(e),
            })

    schema_manifest_df = pd.DataFrame(schema_records)

    schema_manifest_df.to_csv(
        SCHEMA_MANIFEST_PATH,
        index=False
    )

else:
    schema_manifest_df = pd.read_csv(SCHEMA_MANIFEST_PATH)

Next, we summarize the number of unique columns and schema versions by dataset type\. This helps us quickly spot whether a dataset has a stable structure or whether its columns changed across the monthly files\.

In [13]:
schema_file_summary = (
    schema_manifest_df
    .groupby(["dataset_type", "file", "service_month"], dropna=False)
    .agg(
        num_columns=("column_name", "count"),
        schema_signature=("column_name", lambda x: "|".join(sorted(x.dropna())))
    )
    .reset_index()
)

schema_stability_summary = (
    schema_file_summary
    .groupby("dataset_type", dropna=False)
    .agg(
        num_files=("file", "count"),
        min_columns=("num_columns", "min"),
        max_columns=("num_columns", "max"),
        num_schema_versions=("schema_signature", "nunique"),
    )
    .reset_index()
)

schema_stability_summary

,dataset_type,num_files,min_columns,max_columns,num_schema_versions
0,fhv,39,7,7,1
1,fhvhv,39,24,25,2
2,green,39,20,21,2
3,yellow,40,19,20,3


Findings\. A pleasant surprise here is how stable the TLC schemas appear to be\. FHV maintains a single schema across all 39 monthly files, while FHVHV and Green each show only two schema versions\. Yellow Taxi exhibits slightly more variation with three schema versions across the study period, but even that is relatively minor given the number of files involved\. At this stage, there is no evidence of major structural changes that would complicate downstream temporal or spatial standardization\. The next question is whether these schema differences affect core fields that we care about \(timestamps, Taxi Zone IDs, trip metrics\) or whether they are limited to ancillary fields\.

Now let’s inspect the full column inventory for each dataset type\. This gives us a cleaner view of which fields are available in Yellow Taxi, Green Taxi, FHV, and FHVHV before we decide which columns to carry forward\.

In [14]:
column_inventory = (
    schema_manifest_df
    .groupby(["dataset_type", "column_name", "data_type"], dropna=False)
    .agg(
        num_files=("file", "nunique"),
        first_month=("service_month", "min"),
        last_month=("service_month", "max"),
    )
    .reset_index()
    .sort_values(["dataset_type", "column_name", "data_type"])
)

column_inventory

,dataset_type,column_name,data_type,num_files,first_month,last_month
0,fhv,Affiliated_base_number,large_string,38,2023-02,2026-03
1,fhv,Affiliated_base_number,string,1,2023-01,2023-01
2,fhv,DOlocationID,double,1,2023-01,2023-01
3,fhv,DOlocationID,int64,38,2023-02,2026-03
4,fhv,PUlocationID,double,1,2023-01,2023-01
...,...,...,...,...,...,...
99,yellow,tolls_amount,double,40,2023-01,2026-04
100,yellow,total_amount,double,40,2023-01,2026-04
101,yellow,tpep_dropoff_datetime,timestamp[us],40,2023-01,2026-04
102,yellow,tpep_pickup_datetime,timestamp[us],40,2023-01,2026-04


Findings\. The observed schema drift is limited entirely to the January 2023 FHV dataset and reflects differences in column data types rather than structural changes to the schema itself\. All affected columns remain present throughout the study period, with only their underlying parquet representations changing \(for example, \`double\` to \`int64\` and \`string\` to \`large\_string\`\)\. Beginning in February 2023, the FHV schema remains stable through March 2026\. No evidence was found of columns being added, removed, or renamed, suggesting that the TLC datasets maintain highly consistent structures across the analysis period\.

Finally, we create focused views for the fields we expect to matter most later: timestamp fields, Taxi Zone fields, and trip/service metrics\. This is not yet a final column mapping, but it helps us see whether the key ingredients for temporal and spatial standardization are present\.

In [15]:
timestamp_candidates = schema_manifest_df[
    schema_manifest_df["column_name"]
    .str.contains("datetime|timestamp|date|time", case=False, na=False)
].drop_duplicates(
    ["dataset_type", "column_name", "data_type"]
).sort_values(
    ["dataset_type", "column_name"]
)

timestamp_candidates

,file,dataset_type,service_month,column_name,data_type
2,fhv_tripdata_2023-01.parquet,fhv,2023-01,dropOff_datetime,timestamp[us]
1,fhv_tripdata_2023-01.parquet,fhv,2023-01,pickup_datetime,timestamp[us]
279,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,dropoff_datetime,timestamp[us]
277,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,on_scene_datetime,timestamp[us]
278,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,pickup_datetime,timestamp[us]
276,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,request_datetime,timestamp[us]
283,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,trip_time,int64
1226,green_tripdata_2023-01.parquet,green,2023-01,lpep_dropoff_datetime,timestamp[us]
1225,green_tripdata_2023-01.parquet,green,2023-01,lpep_pickup_datetime,timestamp[us]
2021,yellow_tripdata_2023-01.parquet,yellow,2023-01,tpep_dropoff_datetime,timestamp[us]


In [16]:
zone_candidates = schema_manifest_df[
    schema_manifest_df["column_name"]
    .str.contains("location|zone|pu|do", case=False, na=False)
].drop_duplicates(
    ["dataset_type", "column_name", "data_type"]
).sort_values(
    ["dataset_type", "column_name"]
)

zone_candidates

,file,dataset_type,service_month,column_name,data_type
4,fhv_tripdata_2023-01.parquet,fhv,2023-01,DOlocationID,double
11,fhv_tripdata_2023-02.parquet,fhv,2023-02,DOlocationID,int64
3,fhv_tripdata_2023-01.parquet,fhv,2023-01,PUlocationID,double
10,fhv_tripdata_2023-02.parquet,fhv,2023-02,PUlocationID,int64
281,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,DOLocationID,int64
305,fhvhv_tripdata_2023-02.parquet,fhvhv,2023-02,DOLocationID,int32
280,fhvhv_tripdata_2023-01.parquet,fhvhv,2023-01,PULocationID,int64
304,fhvhv_tripdata_2023-02.parquet,fhvhv,2023-02,PULocationID,int32
1230,green_tripdata_2023-01.parquet,green,2023-01,DOLocationID,int64
1250,green_tripdata_2023-02.parquet,green,2023-02,DOLocationID,int32


Findings\. The schema analysis identified consistent temporal and spatial fields across all four TLC datasets\. While naming conventions differ slightly between transportation modes, each dataset contains both pickup and dropoff timestamps as well as pickup and dropoff Taxi Zone identifiers\. FHVHV provides additional temporal fields related to ride requests and driver arrival times, making it the richest temporal dataset in the collection\. Importantly, all TLC datasets already use Taxi Zone IDs as their native spatial reference system, which should significantly simplify downstream spatial standardization compared to the traffic and bus datasets\.

### Summary

Overall, the TLC schemas turned out to be much cleaner than expected\. Aside from a few datatype differences in the January 2023 FHV file, the datasets exhibit very little schema drift across the study period\. More importantly, all four datasets contain consistent pickup and dropoff timestamps along with native Taxi Zone identifiers, giving us the key temporal and spatial fields needed for downstream standardization\. The fact that the TLC datasets already use Taxi Zones as their native geography should make later integration considerably simpler than what we encountered with the traffic and bus datasets\.

## 1\.1\.2\.4 Data inspection, missingness and duplication

Now that we've identified the key temporal and spatial fields across the TLC datasets, let's move from schema\-level QA to record\-level QA\. Before investing time in downstream standardization and aggregation, we want to understand whether the datasets contain missing values, duplicate records, or other issues that could affect later analysis\.

We'll start by previewing a small sample of records from each dataset to build intuition for the data structure and confirm our understanding of the available fields\. From there, we'll evaluate missingness and duplication patterns across the key timestamp and Taxi Zone fields that will ultimately drive the temporal and spatial components of the project\.

In [17]:
# ---------------------------------------------------------------------
# Lightweight dataset preview with DuckDB
# ---------------------------------------------------------------------

sample_files = {
    "yellow": TLC_DATA_DIR / "yellow_tripdata_2023-01.parquet",
    "green": TLC_DATA_DIR / "green_tripdata_2023-01.parquet",
    "fhv": TLC_DATA_DIR / "fhv_tripdata_2023-01.parquet",
    "fhvhv": TLC_DATA_DIR / "fhvhv_tripdata_2023-01.parquet",
}

for dataset_name, path in sample_files.items():
    print("=" * 80)
    print(dataset_name.upper())
    print("=" * 80)

    preview_df = duckdb.sql(
        f"""
        SELECT *
        FROM read_parquet('{path.as_posix()}')
        LIMIT 5
        """
    ).df()

    display(preview_df)

YELLOW


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


GREEN


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,<NA>,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,<NA>,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,<NA>,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,<NA>,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,<NA>,1.0,8.00,1.0,1.0,0.00


FHV


,dispatching_base_num,pickup_datetime,dropOff_datetime,PUlocationID,DOlocationID,SR_Flag,Affiliated_base_number
0,B00008,2023-01-01 00:30:00,2023-01-01 01:00:00,NaN,NaN,<NA>,B00008
1,B00078,2023-01-01 00:01:00,2023-01-01 03:15:00,NaN,NaN,<NA>,B00078
2,B00111,2023-01-01 00:30:00,2023-01-01 01:05:00,NaN,NaN,<NA>,B03406
3,B00112,2023-01-01 00:34:45,2023-01-01 00:52:03,NaN,14.0,<NA>,B00112
4,B00112,2023-01-01 00:11:20,2023-01-01 00:22:03,NaN,14.0,<NA>,B00112


FHVHV


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2023-01-01 00:18:06,2023-01-01 00:19:24,2023-01-01 00:19:38,2023-01-01 00:48:07,48,68,0.94,...,2.30,2.75,0.0,5.22,27.83,N,N,,N,N
1,HV0003,B03404,B03404,2023-01-01 00:48:42,2023-01-01 00:56:20,2023-01-01 00:58:39,2023-01-01 01:33:08,246,163,2.78,...,5.34,2.75,0.0,0.00,50.15,N,N,,N,N
2,HV0003,B03404,B03404,2023-01-01 00:15:35,2023-01-01 00:20:14,2023-01-01 00:20:27,2023-01-01 00:37:54,9,129,8.81,...,2.16,0.00,0.0,0.00,20.22,N,N,,N,N
3,HV0003,B03404,B03404,2023-01-01 00:35:24,2023-01-01 00:39:30,2023-01-01 00:41:05,2023-01-01 00:48:16,129,129,0.67,...,1.22,0.00,0.0,0.00,7.90,N,N,,N,N
4,HV0003,B03404,B03404,2023-01-01 00:43:15,2023-01-01 00:51:10,2023-01-01 00:52:47,2023-01-01 01:04:51,129,92,4.38,...,1.82,0.00,0.0,0.00,16.48,N,N,,N,N


### Data dictionary

Yellow Taxi
VendorID — Taxi technology provider that reported the trip\.
tpep\_pickup\_datetime, tpep\_dropoff\_datetime — Trip pickup and dropoff timestamps\.
passenger\_count — Number of passengers reported for the trip\.
trip\_distance — Trip distance in miles\.
RatecodeID — Fare rate applied to the trip \(standard, JFK, negotiated fare, etc\.\)\.
store\_and\_fwd\_flag — Indicates whether trip data was temporarily stored before transmission\.
PULocationID, DOLocationID — Pickup and dropoff Taxi Zone identifiers\.
payment\_type — Method of payment used by the passenger\.
fare\_amount — Base metered fare\.
extra, mta\_tax, improvement\_surcharge, congestion\_surcharge, Airport\_fee — Additional taxes, fees, and surcharges\.
tip\_amount — Passenger tip amount\.
tolls\_amount — Toll charges incurred during the trip\.
total\_amount — Total amount charged to the passenger\.

Yellow and Green Taxi use largely similar schemas, with different pickup/dropoff timestamp prefixes and a few dataset\-specific fields such as Green’s trip\_type and Yellow’s airport fee field\.

FHV
dispatching\_base\_num — TLC license number of the dispatching base\.
Affiliated\_base\_number — TLC license number of the affiliated base\.
pickup\_datetime, dropOff\_datetime — Trip pickup and dropoff timestamps\.
PUlocationID, DOlocationID — Pickup and dropoff Taxi Zone identifiers\.
SR\_Flag — Shared\-ride indicator\.

FHVHV
hvfhs\_license\_num — TLC license number of the high\-volume for\-hire service \(e\.g\., Uber or Lyft\)\.
dispatching\_base\_num — TLC license number of the dispatching base\.
originating\_base\_num — TLC license number of the originating base\.
request\_datetime — Timestamp when the ride was requested\.
on\_scene\_datetime — Timestamp when the driver arrived at the pickup location\.
pickup\_datetime, dropoff\_datetime — Trip pickup and dropoff timestamps\.
PULocationID, DOLocationID — Pickup and dropoff Taxi Zone identifiers\.
trip\_miles — Trip distance in miles\.
trip\_time — Trip duration in seconds\.
base\_passenger\_fare — Base fare charged to the passenger before additional fees\.
tolls, bcf, sales\_tax, congestion\_surcharge, airport\_fee — Tolls, taxes, and regulatory surcharges\.
tips — Passenger tip amount\.
driver\_pay — Compensation paid to the driver\.
shared\_request\_flag, shared\_match\_flag — Indicators related to shared\-ride participation\.
access\_a\_ride\_flag — Indicates trips associated with the Access\-A\-Ride program\.
wav\_request\_flag, wav\_match\_flag — Indicators related to wheelchair\-accessible vehicle requests and matches\.

### Missingness checks of Spatiotemporal features

Now that we've mapped the important timestamp and Taxi Zone columns, let's scan missingness across the monthly TLC files\. To keep this manageable, we only read the fields needed for this QA pass instead of loading full trip tables into memory\.

In [18]:
# ---------------------------------------------------------------------
# Missingness QA manifest setup
# ---------------------------------------------------------------------

RUN_MISSINGNESS_QA = False

MISSINGNESS_MANIFEST_PATH = QA_DIR / "1.1.2_taxi_fhv_missingness_manifest.csv"

# ---------------------------------------------------------------------
# Define key fields for missingness and duplication QA
# ---------------------------------------------------------------------

qa_fields = {
    "yellow": {
        "pickup_time": "tpep_pickup_datetime",
        "dropoff_time": "tpep_dropoff_datetime",
        "pickup_zone": "PULocationID",
        "dropoff_zone": "DOLocationID",
    },
    "green": {
        "pickup_time": "lpep_pickup_datetime",
        "dropoff_time": "lpep_dropoff_datetime",
        "pickup_zone": "PULocationID",
        "dropoff_zone": "DOLocationID",
    },
    "fhv": {
        "pickup_time": "pickup_datetime",
        "dropoff_time": "dropOff_datetime",
        "pickup_zone": "PUlocationID",
        "dropoff_zone": "DOlocationID",
    },
    "fhvhv": {
        "pickup_time": "pickup_datetime",
        "dropoff_time": "dropoff_datetime",
        "pickup_zone": "PULocationID",
        "dropoff_zone": "DOLocationID",
        "request_time": "request_datetime",
        "on_scene_time": "on_scene_datetime",
    },
}

In [19]:
# ---------------------------------------------------------------------
# Run or load missingness QA with DuckDB + progress logging
# ---------------------------------------------------------------------

if RUN_MISSINGNESS_QA or not MISSINGNESS_MANIFEST_PATH.exists():

    missingness_records = []
    parquet_files = sorted(TLC_DATA_DIR.glob("*.parquet"))
    total_files = len(parquet_files)

    for i, path in enumerate(parquet_files, start=1):
        dataset_type = path.name.split("_tripdata_")[0]
        service_month = path.name.split("_tripdata_")[1].replace(".parquet", "")
        file_size_mb = path.stat().st_size / (1024 * 1024)

        print(
            f"[{i}/{total_files}] Scanning {dataset_type} {service_month} "
            f"({file_size_mb:,.1f} MB): {path.name}",
            flush=True
        )

        field_map = qa_fields[dataset_type]

        try:
            select_parts = ["COUNT(*) AS num_rows"]

            for logical_name, column_name in field_map.items():
                select_parts.append(
                    f'SUM(CASE WHEN "{column_name}" IS NULL THEN 1 ELSE 0 END) '
                    f'AS {logical_name}_missing_count'
                )

            query = f"""
                SELECT
                    {", ".join(select_parts)}
                FROM read_parquet('{path.as_posix()}')
            """

            result = duckdb.sql(query).df().iloc[0].to_dict()

            record = {
                "file": path.name,
                "dataset_type": dataset_type,
                "service_month": service_month,
                "file_size_mb": file_size_mb,
                "error": None,
            }

            record.update(result)

            for logical_name, column_name in field_map.items():
                missing_count = record[f"{logical_name}_missing_count"]
                num_rows = record["num_rows"]

                record[f"{logical_name}_column"] = column_name
                record[f"{logical_name}_missing_rate"] = (
                    missing_count / num_rows if num_rows > 0 else None
                )

            missingness_records.append(record)

            print(
                f"    Done: {int(record['num_rows']):,} rows scanned",
                flush=True
            )

        except Exception as e:
            missingness_records.append({
                "file": path.name,
                "dataset_type": dataset_type,
                "service_month": service_month,
                "file_size_mb": file_size_mb,
                "num_rows": None,
                "error": str(e),
            })

            print(f"    ERROR: {e}", flush=True)

    missingness_df = pd.DataFrame(missingness_records)

    missingness_df.to_csv(
        MISSINGNESS_MANIFEST_PATH,
        index=False
    )

else:
    missingness_df = pd.read_csv(MISSINGNESS_MANIFEST_PATH)

In [20]:
missingness_df = pd.read_csv(MISSINGNESS_MANIFEST_PATH)

print("Rows in manifest:", len(missingness_df))
print("Expected files:", len(list(TLC_DATA_DIR.glob("*.parquet"))))

missingness_df.head()

Rows in manifest: 157
Expected files: 157


,file,dataset_type,service_month,error,num_rows,pickup_time_missing_count,dropoff_time_missing_count,pickup_zone_missing_count,dropoff_zone_missing_count,pickup_time_column,...,pickup_zone_column,pickup_zone_missing_rate,dropoff_zone_column,dropoff_zone_missing_rate,request_time_missing_count,on_scene_time_missing_count,request_time_column,request_time_missing_rate,on_scene_time_column,on_scene_time_missing_rate
0,fhv_tripdata_2023-01.parquet,fhv,2023-01,NaN,1114320.0,0.0,0.0,879428.0,178605.0,pickup_datetime,...,PUlocationID,0.789206,DOlocationID,0.160282,NaN,NaN,NaN,NaN,NaN,NaN
1,fhv_tripdata_2023-02.parquet,fhv,2023-02,NaN,1110797.0,0.0,0.0,875891.0,247884.0,pickup_datetime,...,PUlocationID,0.788525,DOlocationID,0.223159,NaN,NaN,NaN,NaN,NaN,NaN
2,fhv_tripdata_2023-03.parquet,fhv,2023-03,NaN,1328242.0,0.0,0.0,1044255.0,305025.0,pickup_datetime,...,PUlocationID,0.786193,DOlocationID,0.229646,NaN,NaN,NaN,NaN,NaN,NaN
3,fhv_tripdata_2023-04.parquet,fhv,2023-04,NaN,1246479.0,0.0,0.0,983007.0,281166.0,pickup_datetime,...,PUlocationID,0.788627,DOlocationID,0.225568,NaN,NaN,NaN,NaN,NaN,NaN
4,fhv_tripdata_2023-05.parquet,fhv,2023-05,NaN,1385826.0,0.0,0.0,1059872.0,290690.0,pickup_datetime,...,PUlocationID,0.764794,DOlocationID,0.209759,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
missingness_df.groupby("dataset_type").agg({
    "pickup_time_missing_rate": "mean",
    "dropoff_time_missing_rate": "mean",
    "pickup_zone_missing_rate": "mean",
    "dropoff_zone_missing_rate": "mean",
}).round(4)

,pickup_time_missing_rate,dropoff_time_missing_rate,pickup_zone_missing_rate,dropoff_zone_missing_rate
dataset_type,,,,
fhv,0.0,0.0,0.8032,0.1681
fhvhv,0.0,0.0,0.0000,0.0000
green,0.0,0.0,0.0000,0.0000
yellow,0.0,0.0,0.0000,0.0000


Findings: Of the temporal and spatial fields examined, FHV was the only dataset exhibiting substantial missingness\.

In [22]:
missingness_df[
    missingness_df["dataset_type"] == "fhv"
][
    [
        "service_month",
        "pickup_zone_missing_rate",
        "dropoff_zone_missing_rate"
    ]
]

,service_month,pickup_zone_missing_rate,dropoff_zone_missing_rate
0,2023-01,0.789206,0.160282
1,2023-02,0.788525,0.223159
2,2023-03,0.786193,0.229646
3,2023-04,0.788627,0.225568
4,2023-05,0.764794,0.209759
5,2023-06,0.757354,0.143423
6,2023-07,0.768708,0.130148
7,2023-08,0.741556,0.185167
8,2023-09,0.720980,0.150752
9,2023-10,0.757533,0.168620


Findings\. The biggest surprise from the missingness analysis is the FHV dataset\. All four TLC datasets have complete timestamp coverage, and Yellow Taxi, Green Taxi, and FHVHV also have complete Taxi Zone coverage\. FHV is the clear outlier\. Roughly 80% of records are missing a pickup Taxi Zone and about 15–20% are missing a dropoff Taxi Zone\. After checking the monthly trends, this doesn't appear to be a bad file, a bad year, or a schema issue—the missingness is consistently high throughout the entire study period\. For now, we should assume that FHV will be much less useful than the other TLC datasets for Taxi Zone\-level spatial analysis\.

In [23]:
# ---------------------------------------------------------------------
# FHV share and usable spatial coverage
# ---------------------------------------------------------------------

fhv_share_summary = (
    missingness_df
    .groupby("dataset_type", dropna=False)
    .agg(
        total_rows=("num_rows", "sum"),
        pickup_zone_missing_count=("pickup_zone_missing_count", "sum"),
        dropoff_zone_missing_count=("dropoff_zone_missing_count", "sum"),
    )
    .reset_index()
)

fhv_share_summary["trip_share"] = (
    fhv_share_summary["total_rows"]
    / fhv_share_summary["total_rows"].sum()
)

fhv_share_summary["pickup_zone_usable_rows"] = (
    fhv_share_summary["total_rows"]
    - fhv_share_summary["pickup_zone_missing_count"]
)

fhv_share_summary["dropoff_zone_usable_rows"] = (
    fhv_share_summary["total_rows"]
    - fhv_share_summary["dropoff_zone_missing_count"]
)

fhv_share_summary["pickup_zone_usable_rate"] = (
    fhv_share_summary["pickup_zone_usable_rows"]
    / fhv_share_summary["total_rows"]
)

fhv_share_summary["dropoff_zone_usable_rate"] = (
    fhv_share_summary["dropoff_zone_usable_rows"]
    / fhv_share_summary["total_rows"]
)

fhv_share_summary.sort_values("trip_share", ascending=False)

,dataset_type,total_rows,pickup_zone_missing_count,dropoff_zone_missing_count,trip_share,pickup_zone_usable_rows,dropoff_zone_usable_rows,pickup_zone_usable_rate,dropoff_zone_usable_rate
1,fhvhv,778424569.0,0.0,0.0,0.787518,778424569.0,778424569.0,1.000000,1.000000
3,yellow,143080418.0,0.0,0.0,0.144752,143080418.0,143080418.0,1.000000,1.000000
0,fhv,64787450.0,52380541.0,10876684.0,0.065544,12406909.0,53910766.0,0.191502,0.832117
2,green,2160506.0,0.0,0.0,0.002186,2160506.0,2160506.0,1.000000,1.000000


In [24]:
# ---------------------------------------------------------------------
# FHV records with both pickup and dropoff zones present
# ---------------------------------------------------------------------
# FHV is evaluated separately because missing Taxi Zone IDs determine whether
# it can participate in the project grain. The key question is not total FHV
# volume, but how many records have enough pickup/dropoff geography to support
# Taxi Zone-level mobility-state construction.

fhv_both_zone_records = []

for path in sorted(TLC_DATA_DIR.glob("fhv_tripdata_*.parquet")):
    service_month = (
        path.name
        .split("_tripdata_")[1]
        .replace(".parquet", "")
    )

    print(f"Scanning FHV {service_month}: {path.name}", flush=True)

    query = f"""
        SELECT
            COUNT(*) AS num_rows,
            SUM(
                CASE
                    WHEN PUlocationID IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS pickup_zone_usable_rows,
            SUM(
                CASE
                    WHEN DOlocationID IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS dropoff_zone_usable_rows,
            SUM(
                CASE
                    WHEN PUlocationID IS NOT NULL
                     AND DOlocationID IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS both_zones_usable_rows
        FROM read_parquet('{path.as_posix()}')
    """

    result = duckdb.sql(query).df().iloc[0].to_dict()

    result["service_month"] = service_month
    result["file"] = path.name

    fhv_both_zone_records.append(result)

fhv_both_zone_df = pd.DataFrame(fhv_both_zone_records)

fhv_both_zone_df["pickup_zone_usable_rate"] = (
    fhv_both_zone_df["pickup_zone_usable_rows"]
    / fhv_both_zone_df["num_rows"]
)

fhv_both_zone_df["dropoff_zone_usable_rate"] = (
    fhv_both_zone_df["dropoff_zone_usable_rows"]
    / fhv_both_zone_df["num_rows"]
)

fhv_both_zone_df["both_zones_usable_rate"] = (
    fhv_both_zone_df["both_zones_usable_rows"]
    / fhv_both_zone_df["num_rows"]
)

fhv_both_zone_df.head()

Scanning FHV 2023-01: fhv_tripdata_2023-01.parquet
Scanning FHV 2023-02: fhv_tripdata_2023-02.parquet
Scanning FHV 2023-03: fhv_tripdata_2023-03.parquet
Scanning FHV 2023-04: fhv_tripdata_2023-04.parquet
Scanning FHV 2023-05: fhv_tripdata_2023-05.parquet
Scanning FHV 2023-06: fhv_tripdata_2023-06.parquet
Scanning FHV 2023-07: fhv_tripdata_2023-07.parquet
Scanning FHV 2023-08: fhv_tripdata_2023-08.parquet
Scanning FHV 2023-09: fhv_tripdata_2023-09.parquet
Scanning FHV 2023-10: fhv_tripdata_2023-10.parquet
Scanning FHV 2023-11: fhv_tripdata_2023-11.parquet
Scanning FHV 2023-12: fhv_tripdata_2023-12.parquet
Scanning FHV 2024-01: fhv_tripdata_2024-01.parquet
Scanning FHV 2024-02: fhv_tripdata_2024-02.parquet
Scanning FHV 2024-03: fhv_tripdata_2024-03.parquet
Scanning FHV 2024-04: fhv_tripdata_2024-04.parquet
Scanning FHV 2024-05: fhv_tripdata_2024-05.parquet
Scanning FHV 2024-06: fhv_tripdata_2024-06.parquet
Scanning FHV 2024-07: fhv_tripdata_2024-07.parquet
Scanning FHV 2024-08: fhv_tripd

,num_rows,pickup_zone_usable_rows,dropoff_zone_usable_rows,both_zones_usable_rows,service_month,file,pickup_zone_usable_rate,dropoff_zone_usable_rate,both_zones_usable_rate
0,1114320.0,234892.0,935715.0,234650.0,2023-01,fhv_tripdata_2023-01.parquet,0.210794,0.839718,0.210577
1,1110797.0,234906.0,862913.0,232925.0,2023-02,fhv_tripdata_2023-02.parquet,0.211475,0.776841,0.209692
2,1328242.0,283987.0,1023217.0,283811.0,2023-03,fhv_tripdata_2023-03.parquet,0.213807,0.770354,0.213674
3,1246479.0,263472.0,965313.0,263300.0,2023-04,fhv_tripdata_2023-04.parquet,0.211373,0.774432,0.211235
4,1385826.0,325954.0,1095136.0,319926.0,2023-05,fhv_tripdata_2023-05.parquet,0.235206,0.790241,0.230856


In [25]:
fhv_both_zone_summary = pd.DataFrame({
    "num_rows": [fhv_both_zone_df["num_rows"].sum()],
    "pickup_zone_usable_rows": [fhv_both_zone_df["pickup_zone_usable_rows"].sum()],
    "dropoff_zone_usable_rows": [fhv_both_zone_df["dropoff_zone_usable_rows"].sum()],
    "both_zones_usable_rows": [fhv_both_zone_df["both_zones_usable_rows"].sum()],
})

fhv_both_zone_summary["pickup_zone_usable_rate"] = (
    fhv_both_zone_summary["pickup_zone_usable_rows"]
    / fhv_both_zone_summary["num_rows"]
)

fhv_both_zone_summary["dropoff_zone_usable_rate"] = (
    fhv_both_zone_summary["dropoff_zone_usable_rows"]
    / fhv_both_zone_summary["num_rows"]
)

fhv_both_zone_summary["both_zones_usable_rate"] = (
    fhv_both_zone_summary["both_zones_usable_rows"]
    / fhv_both_zone_summary["num_rows"]
)

fhv_both_zone_summary

,num_rows,pickup_zone_usable_rows,dropoff_zone_usable_rows,both_zones_usable_rows,pickup_zone_usable_rate,dropoff_zone_usable_rate,both_zones_usable_rate
0,64787450.0,12406909.0,53910766.0,12185451.0,0.191502,0.832117,0.188084


Findings\. Before excluding FHV from the project, we quantified both its share of overall TLC activity and the fraction of trips that remain usable for Taxi Zone\-level analysis\. FHV accounts for only 6\.6% of all TLC trips during the study period, compared with 78\.8% for FHVHV and 14\.5% for Yellow Taxi\. More importantly, only 19\.2% of FHV trips contain a valid pickup Taxi Zone, and only 18\.8% contain both pickup and dropoff Taxi Zones\. As a result, the subset of FHV trips that can fully participate in Taxi Zone × Date × Temporal Bucket mobility\-state construction represents only about 1\.2% of the overall TLC trip universe\. 

Given the project's emphasis on Taxi Zone\-level mobility states, anomaly detection, clustering, and multimodal integration, the analytical value of retaining FHV appears limited relative to the additional complexity it introduces\. Based on these findings, we will exclude FHV from the project's core mobility\-state pipeline and focus on Yellow Taxi, Green Taxi, and FHVHV, which provide substantially more complete and spatially reliable coverage\.

📝 Note: FHV will be excluded from downstream mobility\-state analysis due to extensive Taxi Zone missingness\.

### Summary

The spatiotemporal missingness analysis was largely reassuring\. All four TLC datasets exhibit complete coverage for their primary timestamp fields, and Yellow Taxi, Green Taxi, and FHVHV also exhibit complete coverage for both pickup and dropoff Taxi Zone identifiers\. This gives us high confidence that these datasets can support the Taxi Zone × Date × Temporal Bucket mobility tables that serve as the foundation for the rest of the project\. The clear outlier is the FHV dataset, where roughly 80% of records are missing pickup Taxi Zones and 15–20% are missing dropoff Taxi Zones throughout the entire study period\. Because the project's unsupervised learning, anomaly detection, mobility regime clustering, and social\-integration workflows are all built around Taxi Zone\-level mobility states, FHV's limited spatial coverage significantly reduces its analytical value\. Rather than carrying this limitation through the remainder of the pipeline, we will focus the project's core mobility modeling efforts on Yellow Taxi, Green Taxi, and FHVHV, which provide complete and reliable spatiotemporal coverage\.

### Missingness checks of mobility features

We first evaluated missingness in the temporal and spatial fields required to build the project's canonical mobility tables\. Having confirmed those fields are largely complete, we now examine the mobility and fare metrics that may later be incorporated into anomaly detection and mobility regime clustering\.

In [26]:
# ---------------------------------------------------------------------
# Define mobility feature fields for missingness QA
# Mobility feature missingness QA manifest setup
# ---------------------------------------------------------------------

RUN_MOBILITY_MISSINGNESS_QA = False

MOBILITY_MISSINGNESS_MANIFEST_PATH = (
    QA_DIR / "1.1.2_taxi_fhv_mobility_feature_missingness_manifest.csv"
)

mobility_feature_fields = {
    "yellow": {
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
    },
    "green": {
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
    },
    "fhvhv": {
        "trip_miles": "trip_miles",
        "trip_time": "trip_time",
    },
}

mobility_feature_fields

{'yellow': {'passenger_count': 'passenger_count',
  'trip_distance': 'trip_distance'},
 'green': {'passenger_count': 'passenger_count',
  'trip_distance': 'trip_distance'},
 'fhvhv': {'trip_miles': 'trip_miles', 'trip_time': 'trip_time'}}

Let's evaluate missingness in the core mobility metrics that may later be incorporated into feature engineering, anomaly detection, and mobility regime clustering workflows\. As before, we'll persist the results to a QA manifest so the scan only needs to run once\.

In [27]:
# ---------------------------------------------------------------------
# Run or load mobility feature missingness QA
# ---------------------------------------------------------------------

if (
    RUN_MOBILITY_MISSINGNESS_QA
    or not MOBILITY_MISSINGNESS_MANIFEST_PATH.exists()
):

    mobility_missingness_records = []

    parquet_files = sorted(TLC_DATA_DIR.glob("*.parquet"))

    for path in parquet_files:

        dataset_type = path.name.split("_tripdata_")[0]

        if dataset_type not in mobility_feature_fields:
            continue

        service_month = (
            path.name
            .split("_tripdata_")[1]
            .replace(".parquet", "")
        )

        field_map = mobility_feature_fields[dataset_type]

        try:

            select_parts = [
                "COUNT(*) AS num_rows"
            ]

            for logical_name, column_name in field_map.items():

                select_parts.append(
                    f'''
                    SUM(
                        CASE
                            WHEN "{column_name}" IS NULL
                            THEN 1
                            ELSE 0
                        END
                    ) AS {logical_name}_missing_count
                    '''
                )

            query = f"""
                SELECT
                    {",".join(select_parts)}
                FROM read_parquet(
                    '{path.as_posix()}'
                )
            """

            result = duckdb.sql(query).df().iloc[0].to_dict()

            record = {
                "file": path.name,
                "dataset_type": dataset_type,
                "service_month": service_month,
                "error": None,
            }

            record.update(result)

            for logical_name, column_name in field_map.items():

                record[f"{logical_name}_column"] = column_name

                record[f"{logical_name}_missing_rate"] = (
                    record[f"{logical_name}_missing_count"]
                    / record["num_rows"]
                )

            mobility_missingness_records.append(record)

        except Exception as e:

            mobility_missingness_records.append({
                "file": path.name,
                "dataset_type": dataset_type,
                "service_month": service_month,
                "error": str(e),
            })

    mobility_missingness_df = pd.DataFrame(
        mobility_missingness_records
    )

    mobility_missingness_df.to_csv(
        MOBILITY_MISSINGNESS_MANIFEST_PATH,
        index=False
    )

else:

    mobility_missingness_df = pd.read_csv(
        MOBILITY_MISSINGNESS_MANIFEST_PATH
    )

In [28]:
# ---------------------------------------------------------------------
# Summarize mobility feature missingness by dataset
# ---------------------------------------------------------------------

mobility_missingness_summary = (
    mobility_missingness_df
    .groupby("dataset_type", dropna=False)
    .agg(
        num_files=("file", "count"),
        total_rows=("num_rows", "sum"),
        avg_passenger_count_missing_rate=("passenger_count_missing_rate", "mean"),
        avg_trip_distance_missing_rate=("trip_distance_missing_rate", "mean"),
        avg_trip_miles_missing_rate=("trip_miles_missing_rate", "mean"),
        avg_trip_time_missing_rate=("trip_time_missing_rate", "mean"),
        max_passenger_count_missing_rate=("passenger_count_missing_rate", "max"),
        max_trip_distance_missing_rate=("trip_distance_missing_rate", "max"),
        max_trip_miles_missing_rate=("trip_miles_missing_rate", "max"),
        max_trip_time_missing_rate=("trip_time_missing_rate", "max"),
    )
    .reset_index()
)

mobility_missingness_summary

,dataset_type,num_files,total_rows,avg_passenger_count_missing_rate,avg_trip_distance_missing_rate,avg_trip_miles_missing_rate,avg_trip_time_missing_rate,max_passenger_count_missing_rate,max_trip_distance_missing_rate,max_trip_miles_missing_rate,max_trip_time_missing_rate
0,fhvhv,39,778424569.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0
1,green,39,2160506.0,0.070256,0.0,NaN,NaN,0.151375,0.0,NaN,NaN
2,yellow,40,143080418.0,0.136651,0.0,NaN,NaN,0.300987,0.0,NaN,NaN


Findings\. The mobility\-metric missingness analysis was largely uneventful, which is exactly what we were hoping to see\. The core distance and duration fields that are most likely to be used later for feature engineering, anomaly detection, and mobility regime clustering exhibit complete coverage across the Yellow Taxi, Green Taxi, and FHVHV datasets\. The only meaningful missingness appears in the passenger\-count fields, averaging roughly 7% in Green Taxi and 14% in Yellow Taxi, with a handful of months exhibiting higher rates\. Because passenger count is not currently expected to be a primary mobility\-state feature, this issue is unlikely to materially affect the downstream analyses\. Overall, the mobility metrics that appear most relevant to the project's unsupervised learning and forecasting workflows are complete and ready to be incorporated into later stages of the pipeline\.

### Duplication assessment

In [29]:
# ---------------------------------------------------------------------
# Confirm that retained TLC datasets do not share a universal trip ID
# ---------------------------------------------------------------------

retained_dataset_types = ["yellow", "green", "fhvhv"]

retained_schema_columns = (
    schema_manifest_df[
        schema_manifest_df["dataset_type"].isin(retained_dataset_types)
    ]
    .groupby("dataset_type")["column_name"]
    .apply(lambda x: sorted(set(x.dropna())))
    .to_dict()
)

for dataset_type, columns in retained_schema_columns.items():
    id_like_columns = [
        col for col in columns
        if "id" in col.lower() or "license" in col.lower() or "base" in col.lower()
    ]

    print("=" * 80)
    print(dataset_type.upper())
    print("=" * 80)
    print(id_like_columns)

FHVHV
['DOLocationID', 'PULocationID', 'access_a_ride_flag', 'base_passenger_fare', 'dispatching_base_num', 'hvfhs_license_num', 'originating_base_num']
GREEN
['DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID']
YELLOW
['DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID']


Findings\. The TLC datasets do not provide a universal trip identifier that would support straightforward duplicate\-record detection\. Given the scale of the data and the project's focus on aggregated mobility states rather than individual trips, we do not perform a dedicated duplicate\-record analysis during 1\.1\.2\. Potential duplication issues will be evaluated indirectly during aggregation and mobility\-table QA in later stages of the pipeline\.

## 1\.1\.2\.5 Categorical distribution validation

Let's take a quick look at a few high\-level structural distributions within the retained datasets\. The goal here is simply to confirm that the TLC data exhibits broad geographic coverage and does not contain any obvious gaps or unexpected patterns that could affect downstream aggregation into Taxi Zone × Date × Temporal Bucket mobility states\.

### Taxi Zone Coverage

Because Taxi Zones are the project's canonical geography, the most useful distribution check is understanding how broadly trips are distributed across the Taxi Zone system\. Rather than focusing on individual neighborhoods, we use the number of unique pickup and dropoff zones as a simple measure of geographic coverage across the city\.

In [30]:
# ---------------------------------------------------------------------
# Taxi Zone coverage QA manifest setup
# ---------------------------------------------------------------------

RUN_TAXI_ZONE_COVERAGE_QA = False

TAXI_ZONE_COVERAGE_MANIFEST_PATH = (
    QA_DIR / "1.1.2_tlc_taxi_zone_coverage_manifest.csv"
)

In [31]:
# ---------------------------------------------------------------------
# Run or load Taxi Zone coverage QA
# ---------------------------------------------------------------------

if (
    RUN_TAXI_ZONE_COVERAGE_QA
    or not TAXI_ZONE_COVERAGE_MANIFEST_PATH.exists()
):

    zone_coverage_records = []

    zone_fields = {
        "yellow": ("PULocationID", "DOLocationID"),
        "green": ("PULocationID", "DOLocationID"),
        "fhvhv": ("PULocationID", "DOLocationID"),
    }

    for dataset_type, (pu_col, do_col) in zone_fields.items():

        print(f"Scanning Taxi Zone coverage for {dataset_type}...", flush=True)

        query = f"""
            WITH pickup_counts AS (
                SELECT
                    '{dataset_type}' AS dataset_type,
                    'pickup' AS zone_role,
                    {pu_col} AS taxi_zone_id,
                    COUNT(*) AS trips
                FROM read_parquet(
                    '{(TLC_DATA_DIR / f"{dataset_type}_tripdata_*.parquet").as_posix()}'
                )
                GROUP BY 1, 2, 3
            ),

            dropoff_counts AS (
                SELECT
                    '{dataset_type}' AS dataset_type,
                    'dropoff' AS zone_role,
                    {do_col} AS taxi_zone_id,
                    COUNT(*) AS trips
                FROM read_parquet(
                    '{(TLC_DATA_DIR / f"{dataset_type}_tripdata_*.parquet").as_posix()}'
                )
                GROUP BY 1, 2, 3
            )

            SELECT *
            FROM pickup_counts

            UNION ALL

            SELECT *
            FROM dropoff_counts
        """

        result_df = duckdb.sql(query).df()

        zone_coverage_records.append(result_df)

        print(
            f"Done: {dataset_type} "
            f"({len(result_df):,} zone-role rows)",
            flush=True
        )

    taxi_zone_coverage_df = pd.concat(
        zone_coverage_records,
        ignore_index=True
    )

    taxi_zone_coverage_df.to_csv(
        TAXI_ZONE_COVERAGE_MANIFEST_PATH,
        index=False
    )

else:

    taxi_zone_coverage_df = pd.read_csv(
        TAXI_ZONE_COVERAGE_MANIFEST_PATH
    )

In [32]:
# ---------------------------------------------------------------------
# Unique Taxi Zone coverage summary
# ---------------------------------------------------------------------

taxi_zone_coverage_summary = (
    taxi_zone_coverage_df
    .groupby(["dataset_type", "zone_role"], dropna=False)
    .agg(
        unique_zones=("taxi_zone_id", "nunique"),
        total_trips=("trips", "sum"),
    )
    .reset_index()
)

taxi_zone_coverage_summary

,dataset_type,zone_role,unique_zones,total_trips
0,fhvhv,dropoff,264,778424569
1,fhvhv,pickup,263,778424569
2,green,dropoff,260,2160506
3,green,pickup,258,2160506
4,yellow,dropoff,263,143080418
5,yellow,pickup,263,143080418


Findings\. The Taxi Zone coverage results look exactly like what we'd hope to see\. Yellow Taxi and FHVHV reach essentially the entire Taxi Zone system, while Green Taxi reaches slightly fewer zones, which is expected given its much smaller trip volume\. Overall, the retained TLC datasets exhibit excellent geographic coverage and appear well suited for Taxi Zone\-level aggregation, anomaly detection, clustering, and mobility\-state analysis\. One minor item remains for follow\-up: FHVHV reports 264 unique dropoff zones even though the official Taxi Zone system contains 263 zones\. We'll investigate and resolve that discrepancy during 1\.2\.2 when TLC records are validated against the Taxi Zone reference layer\.

### Operator distribution checks

In [33]:
# ---------------------------------------------------------------------
# FHVHV operator distribution QA setup
# ---------------------------------------------------------------------

RUN_OPERATOR_DISTRIBUTION_QA = False

OPERATOR_DISTRIBUTION_MANIFEST_PATH = (
    QA_DIR / "1.1.2_fhvhv_operator_distribution_manifest.csv"
)

In [34]:
# ---------------------------------------------------------------------
# Run or load FHVHV operator distribution QA
# ---------------------------------------------------------------------

if (
    RUN_OPERATOR_DISTRIBUTION_QA
    or not OPERATOR_DISTRIBUTION_MANIFEST_PATH.exists()
):

    query = f"""
        SELECT
            hvfhs_license_num,
            COUNT(*) AS trips
        FROM read_parquet(
            '{(TLC_DATA_DIR / "fhvhv_tripdata_*.parquet").as_posix()}'
        )
        GROUP BY 1
        ORDER BY trips DESC
    """

    operator_distribution_df = duckdb.sql(query).df()

    operator_distribution_df["trip_share"] = (
        operator_distribution_df["trips"]
        / operator_distribution_df["trips"].sum()
    )

    operator_distribution_df.to_csv(
        OPERATOR_DISTRIBUTION_MANIFEST_PATH,
        index=False
    )

else:

    operator_distribution_df = pd.read_csv(
        OPERATOR_DISTRIBUTION_MANIFEST_PATH
    )

In [35]:
operator_distribution_df

,hvfhs_license_num,trips,trip_share
0,HV0003,567611441,0.72918
1,HV0005,210813128,0.27082


Findings\. The FHVHV dataset is composed entirely of trips dispatched by Uber and Lyft, the two licensed high\-volume for\-hire vehicle operators in New York City\. Uber accounts for roughly 73% of all FHVHV trips during the study period, while Lyft accounts for the remaining 27%\. This distribution appears reasonable and confirms that the dataset is dominated by the two rideshare platforms that currently make up the overwhelming majority of New York City's high\-volume for\-hire vehicle market\.

## Close

The TLC QA process surfaced far fewer issues than expected\. The retained Yellow Taxi, Green Taxi, and FHVHV datasets exhibit stable schemas, complete temporal coverage, strong Taxi Zone coverage, and minimal missingness in the fields most relevant to mobility analysis\. The one notable exception was the FHV dataset, whose extensive Taxi Zone missingness substantially limits its usefulness for the project's Taxi Zone × Date × Temporal Bucket framework\. As a result, FHV will be excluded from the core mobility\-state pipeline\. With the remaining datasets validated and key fields identified, we can now move on to temporal and spatial standardization, where individual trip records will begin to be transformed into the multimodal mobility tables used throughout the rest of the project\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>